In [14]:
import numpy as np
import jax
import jax.numpy as jnp


def _build_kedges(mesh_shape, box_shape, kedges):
    """
    Build kedges in a way that is jit-safe.

    mesh_shape, box_shape: Python sequences (e.g. mesh.shape, (Lx, Ly, Lz))
    kedges:
      - None  -> auto spacing (twice fundamental)
      - int   -> number of bins (spacing from kmax/(n+1))
      - float -> bin width
      - array-like (np/jnp) -> used as is

    This function only uses NumPy for the auto cases, so it never sees JAX tracers.
    """
    # Turn shapes into NumPy arrays of floats for kmax computation
    mesh_shape_np = np.asarray(mesh_shape, dtype=float)
    box_shape_np = np.asarray(box_shape, dtype=float)

    kmax = np.pi * np.min(mesh_shape_np / box_shape_np)

    # Auto cases: kedges is None / int / float (all static under jit)
    if kedges is None:
        # twice fundamental mode of smallest box side
        dk = 2.0 * np.pi / np.min(box_shape_np) * 2.0
        edges = np.arange(dk, kmax, dk) + 0.5 * dk
        if edges.size < 2:
            edges = np.linspace(kmax / 4.0, 0.9 * kmax, 2)
        return jnp.asarray(edges)

    if isinstance(kedges, int):
        dk = kmax / (kedges + 1)
        edges = np.arange(dk, kmax, dk) + 0.5 * dk
        if edges.size < 2:
            edges = np.linspace(kmax / 4.0, 0.9 * kmax, 2)
        return jnp.asarray(edges)

    if isinstance(kedges, float):
        dk = float(kedges)
        edges = np.arange(dk, kmax, dk) + 0.5 * dk
        if edges.size < 2:
            edges = np.linspace(kmax / 4.0, 0.9 * kmax, 2)
        return jnp.asarray(edges)

    # Otherwise: treat kedges as array-like (np/jnp)
    return jnp.asarray(kedges)


def _initialize_pk(mesh_shape, box_shape, kedges, los):
    """
    JIT-friendly k grid / binning.

    mesh_shape, box_shape: Python sequences (not JAX arrays)
    kedges: 1D jax array (already built)
    los: None or length-3 (Python or JAX)
    """
    mesh_shape = tuple(int(m) for m in mesh_shape)
    box_shape = tuple(float(L) for L in box_shape)

    kedges = jnp.asarray(kedges)
    ndim = len(mesh_shape)

    # Build k grid
    kvec = []
    for axis, (m, L) in enumerate(zip(mesh_shape, box_shape)):
        k1d = 2.0 * jnp.pi * m / L * jnp.fft.fftfreq(m)
        shape = [1] * ndim
        shape[axis] = m
        kvec.append(k1d.reshape(shape))

    kmesh = jnp.sqrt(sum(ki**2 for ki in kvec))

    # Bin indices and counts
    dig = jnp.digitize(kmesh.reshape(-1), kedges)
    nbins = kedges.shape[0] + 1

    kcount = jnp.bincount(dig, length=nbins)
    ksum = jnp.bincount(dig, weights=kmesh.reshape(-1), length=nbins)
    kavg = ksum / jnp.where(kcount == 0, 1, kcount)
    kavg = kavg[1:-1]  # same as your original: drop the first/last bins

    # μ mesh
    if los is None:
        mumesh = None
    else:
        los = jnp.asarray(los)
        mumesh = sum(ki * los_i for ki, los_i in zip(kvec, los))
        kmesh_safe = jnp.where(kmesh == 0, 1, kmesh)
        mumesh = jnp.where(kmesh == 0, 0, mumesh / kmesh_safe)

    return dig, kcount, kavg, mumesh, kedges


def _legendre(mu, ell: int):
    if ell == 0:
        return jnp.ones_like(mu)
    if ell == 1:
        return mu
    P0 = jnp.ones_like(mu)
    P1 = mu
    for n in range(2, ell + 1):
        Pn = ((2 * n - 1) * mu * P1 - (n - 1) * P0) / n
        P0, P1 = P1, Pn
    return P1


def power_spec_3d(
    mesh,
    mesh2=None,
    *,
    box_shape=None,
    kedges=None,
    multipoles=0,
    los=(0.0, 0.0, 1.0),
):
    """
    3D power spectrum (auto/cross) with dynamic kedges and jit compatibility.

    kedges:
      - None / int / float / array-like, as in your original semantics.
      - Can vary between calls; JAX will just recompile if its length changes.
    """
    mesh_shape = mesh.shape
    if box_shape is None:
        box_shape = mesh_shape
    else:
        box_shape = tuple(box_shape)

    # Make multipoles static Python ints
    if np.ndim(multipoles) == 0:
        poles = (int(multipoles),)
        return_scalar = True
    else:
        poles = tuple(int(e) for e in np.asarray(multipoles))
        return_scalar = False

    # Decide if we actually need LOS
    if any(ell > 0 for ell in poles):
        los_arr = np.asarray(los, dtype=float)
        los_arr = los_arr / np.linalg.norm(los_arr)
        los_tuple = tuple(float(x) for x in los_arr)
    else:
        los_tuple = None

    # Build kedges (this handles None/int/float/array and is jit-safe)
    kedges_jax = _build_kedges(mesh_shape, box_shape, kedges)

    # FFT(s)
    meshk = jnp.fft.fftn(mesh, norm="ortho")
    if mesh2 is None:
        mmk = meshk.real**2 + meshk.imag**2
    else:
        meshk2 = jnp.fft.fftn(mesh2, norm="ortho")
        mmk = meshk * meshk2.conj()

    # k-bin setup
    dig, kcount, kavg, mumesh, kedges_jax = _initialize_pk(
        mesh_shape, box_shape, kedges_jax, los_tuple
    )
    n_bins = kedges_jax.shape[0] + 1
    norm = jnp.where(kcount > 0, kcount, 1)

    # Multipole loop
    pk_rows = []
    for ell in poles:
        if ell == 0 or mumesh is None:
            w_ell = jnp.ones_like(mmk.real)
        else:
            w_ell = _legendre(mumesh, ell)

        weights = (mmk * (2 * ell + 1) * w_ell).reshape(-1)

        if mesh2 is None:
            psum = jnp.bincount(dig, weights=weights, length=n_bins)
        else:
            # keep your original cross-spectrum magnitude behaviour
            psum_real = jnp.bincount(dig, weights=weights.real, length=n_bins)
            psum_imag = jnp.bincount(dig, weights=weights.imag, length=n_bins)
            psum = jnp.sqrt(psum_real**2 + psum_imag**2)

        pk_rows.append(psum)

    pk = jnp.stack(pk_rows, axis=0)
    pk = (pk / norm)[..., 1:-1] * (
        jnp.array(box_shape, dtype=float) / jnp.array(mesh_shape, dtype=float)
    ).prod()

    if return_scalar:
        return kavg, pk[0]
    return kavg, pk


In [15]:
jitted_ps = jax.jit(
    power_spec_3d,
    static_argnames=("box_shape", "multipoles", "los"),
)

# kedges dynamic (None, int, float, or array)
k, P0 = jitted_ps(mesh, box_shape=(Lx, Ly, Lz), kedges=None, multipoles=0)
k, P02 = jitted_ps(mesh, box_shape=(Lx, Ly, Lz), kedges=32, multipoles=(0, 2))
k, P0dyn = jitted_ps(mesh, box_shape=(Lx, Ly, Lz),
                     kedges=jnp.linspace(kmin, kmax, 40),
                     multipoles=0)


ValueError: digitize: bins must be a 1-dimensional array; got bins=JitTracer<~int32[]>

In [12]:
k

Array([0.19317487, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        ], dtype=float32)